In [ ]:
# Dofbot Gripper Action Server

Este paquete implementa un **Action Server** de ROS 2 para controlar el gripper del robot **Dofbot**.  
Permite enviar metas que especifican una **posición del gripper** y una **duración**, y recibe retroalimentación en tiempo real durante el movimiento.

## Funcionalidades
- Validación de parámetros (estado y duración).
- Publicación de feedback cada segundo.
- Manejo de errores y cancelación.

In [ ]:
## Requisitos

- ROS 2 Jazzy instalado.
- Paquete `dofbot_interfaces` con la acción personalizada `GripperCmd`.
- Python 3.8+ y las dependencias listadas.

## Instalación

```bash
# Clonar el repositorio en tu workspace de ROS 2
cd ~/ROS2Dev/dofbotx/src
git clone https://github.com/arrg-mx/tsr-2026-2.git
cd ~/ROS2Dev/dofbotx
colcon build --packages-select dofbot_control
source install/setup.bash

In [ ]:

#### **Celda 3: Código completo del servidor**

En esta celda pegas el script del servidor. Puedes ejecutarlo directamente si el entorno tiene ROS 2, pero normalmente se corre en terminal aparte. Para el notebook, lo dejamos como muestra:

```python
# cell 3: código del action server (se puede mostrar, no se ejecutará en el notebook porque requiere rclpy.spin)
import rclpy
from rclpy.node import Node
from rclpy.action import ActionServer
from rclpy.action.server import ServerGoalHandle
from dofbot_interfaces.action import GripperCmd
import uuid
import time

class DofbotSimpleActionSrv(Node):
    def __init__(self, node_name):
        super().__init__(node_name)
        self._action_srv = ActionServer(
            self,
            GripperCmd,
            'gripper_command',
            self.__execute_callback
        )
        self.get_logger().info(f"Dofbot ActionServer {node_name} inicializado.")

    def __validate_range(self, value, min_val, max_val, strict=False):
        if strict:
            if value <= max_val and value >= min_val:
                return False
        else:        
            if value < max_val and value > min_val:
                return False
        return True

    def __execute_callback(self, goal_handle: ServerGoalHandle):
        goal_id = uuid.UUID(bytes=bytes(goal_handle.goal_id.uuid))
        self.get_logger().info(f"--> Recibimos una nueva GOAL: GOAL_ID({str(goal_id)})")
        gripper_goal = goal_handle.request
        goal_state = gripper_goal.gripper_state
        goal_duration = gripper_goal.duration

        if (self.__validate_range(goal_state, GripperCmd.Goal.OPEN, GripperCmd.Goal.CLOSE)):
            goal_handle.abort()
            self.get_logger().warn(f"--> GOAL_ID({str(goal_id)}) was ABORTED for 'GRIPPER out of range' rule.")
            result = GripperCmd.Result()
            result.success = False
            result.string_status_message = f"ERROR: GRIPPER_STATE {goal_state} debe ser menor a {GripperCmd.Goal.CLOSE} y mayor a {GripperCmd.Goal.OPEN}."
            result.current_state = 0.0
            return result
        if (self.__validate_range(goal_duration, 0.0, 10.0) ):
            goal_handle.abort()
            self.get_logger().warn(f"--> GOAL_ID({str(goal_id)}) was ABORTED for 'DURATION out of range' rule.")
            result = GripperCmd.Result()
            result.success = False
            result.string_status_message = f"ERROR: DURATION {goal_duration} debe ser mayor a {0.0} y menor a {10.0}."
            result.current_state = 0.0
            return result

        igripper_state = -0.7045
        delta = (goal_state - igripper_state) / int(goal_duration)
        start_time = time.time()
        self.get_logger().info(f"--> Executing GOAL_ID({str(goal_id)}).")
        while int(time.time() - start_time) < int(goal_duration):
            feedback_msg = GripperCmd.Feedback()
            feedback_msg.current_state = igripper_state
            igripper_state += delta
            goal_handle.publish_feedback(feedback_msg)
            time.sleep(1.0)

        self.get_logger().info(f"--> Finish GOAL_ID({str(goal_id)}) successfully.")
        goal_handle.succeed()
        result = GripperCmd.Result()
        result.current_state = goal_state
        result.success = True
        result.string_status_message = "Gripper move successfully."
        return result

def main(args=None):
    rclpy.init(args=args)
    node = DofbotSimpleActionSrv('gripper_action_srv_node')
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        rclpy.shutdown()

if __name__ == "__main__":
    main()

In [ ]:
## Explicación del código

### Validación de metas
- La meta se rechaza (`abort`) si `gripper_state` está fuera de `OPEN`/`CLOSE` o si `duration` no está entre 0 y 10 segundos.

### Simulación del movimiento
- Se usa un bucle que se ejecuta durante la duración solicitada.
- En cada iteración (1 segundo) se publica un `Feedback` con el estado actual calculado linealmente.

### Finalización
- Al terminar el bucle se llama `goal_handle.succeed()` y se devuelve un `Result` exitoso.

In [ ]:
## Ejecutar el servidor de acciones

Desde una terminal con ROS 2 configurado:

```bash
ros2 run dofbot_actions gripper_action_server.py

In [ ]:

#### **Celda 6: Cliente de prueba (opcional)**

Para probar el servidor, puedes incluir un cliente simple.

```python
# cell 6: Ejemplo de cliente (ejecutar en otra terminal o en un script)
import rclpy
from rclpy.action import ActionClient
from rclpy.node import Node
from dofbot_interfaces.action import GripperCmd

class GripperClient(Node):
    def __init__(self):
        super().__init__('gripper_client')
        self._client = ActionClient(self, GripperCmd, 'gripper_command')

    def send_goal(self, state, duration):
        if not self._client.wait_for_server(timeout_sec=5.0):
            self.get_logger().error('Servidor no disponible')
            return
        goal = GripperCmd.Goal()
        goal.gripper_state = state
        goal.duration = duration
        self.get_logger().info('Enviando meta...')
        self._send_goal_future = self._client.send_goal_async(
            goal, feedback_callback=self.feedback_callback
        )
        self._send_goal_future.add_done_callback(self.goal_response_callback)

    def goal_response_callback(self, future):
        goal_handle = future.result()
        if not goal_handle.accepted:
            self.get_logger().info('Meta rechazada')
            return
        self.get_logger().info('Meta aceptada, esperando resultado...')
        self._get_result_future = goal_handle.get_result_async()
        self._get_result_future.add_done_callback(self.get_result_callback)

    def feedback_callback(self, feedback_msg):
        self.get_logger().info(f'Feedback: {feedback_msg.feedback.current_state}')

    def get_result_callback(self, future):
        result = future.result().result
        if result.success:
            self.get_logger().info(f'Éxito: {result.string_status_message}')
        else:
            self.get_logger().info(f'Fallo: {result.string_status_message}')
        rclpy.shutdown()

if __name__ == '__main__':
    rclpy.init()
    client = GripperClient()
    # Enviar meta: estado = GripperCmd.Goal.CLOSE (por ej. 0.8), duración 5 s
    client.send_goal(0.8, 5.0)  # Ajusta según tus constantes
    rclpy.spin(client)

In [ ]:
## Mejoras futuras
- Reemplazar la simulación por la lectura real del estado del gripper.
- Permitir cancelación de la meta usando `goal_handle.is_cancel_requested`.
- Añadir logs más detallados con ROS 2 params.